# 03 — Bayesian Probability Modeling

This notebook estimates the probability of finding a graduate-degree holder aged ≤17 within a geographic region, using a Bayesian Poisson model with uncertainty quantification.

**Sections:**
1. Model setup & prior elicitation
2. National base rate estimation
3. Municipality-level estimates
4. Radius query: Danish Embassy, CDMX
5. Sensitivity analysis
6. Export results

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import scipy.stats as stats
from IPython.display import display

from src.model import (
    ProdigyProbabilityModel,
    ModelConfig,
    sensitivity_analysis,
    NATIONAL_BASE_RATE_PRIOR_ALPHA,
    NATIONAL_BASE_RATE_PRIOR_BETA,
    TOTAL_GRADUATE_COMPLETIONS_MX_2022,
)
from src.geo_utils import DANISH_EMBASSY_CDMX

plt.style.use('seaborn-v0_8-whitegrid')
SEED = 42
rng = np.random.default_rng(SEED)

## 1. Prior Elicitation

We model the national base rate `p` (probability any individual is a sub-18 graduate) as:

$$p \sim \text{Beta}(\alpha, \beta)$$

**Prior reasoning:**
- ANUIES reports ~160,000 graduate completions/year in Mexico
- We estimate 0–10 completions under age 18 nationally per year
- This implies base rate ≈ 4–6 per 100,000 graduate completions
- Relative to total population: ~1–5 per 10,000,000

In [ ]:
# Plot the Beta prior
alpha = NATIONAL_BASE_RATE_PRIOR_ALPHA
beta = NATIONAL_BASE_RATE_PRIOR_BETA
prior_mean = alpha / (alpha + beta)
prior_std = np.sqrt(alpha * beta / ((alpha + beta)**2 * (alpha + beta + 1)))

x = np.linspace(0, prior_mean * 10, 1000)
y = stats.beta.pdf(x, alpha, beta)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(x, y, color='#2a9d8f', linewidth=2)
ax.fill_between(x, y, alpha=0.3, color='#2a9d8f')
ax.axvline(prior_mean, color='#e76f51', linestyle='--', label=f'Prior mean: {prior_mean:.2e}')
ax.set_xlabel('Base rate p (prodigy per person)')
ax.set_ylabel('Density')
ax.set_title(f'Beta({alpha}, {beta}) Prior on National Prodigy Base Rate')
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/maps/prior_distribution.png', dpi=150)
plt.show()

print(f"Prior mean:  {prior_mean:.2e}")
print(f"Prior std:   {prior_std:.2e}")
print(f"Implied N in Mexico: {prior_mean * 126_014_024:.2f}")

## 2. National Base Rate Estimation

In [ ]:
model = ProdigyProbabilityModel(ModelConfig(n_samples=50_000))
samples = model.sample_base_rate()

fig = go.Figure()
fig.add_trace(go.Histogram(
    x=samples * 1e6,
    nbinsx=80,
    marker_color='#457b9d',
    opacity=0.8,
    name='Posterior samples'
))
fig.update_layout(
    title='Sampled Base Rate Distribution (per million population)',
    xaxis_title='Base rate (prodigies per million)',
    yaxis_title='Count',
    template='plotly_white',
)
fig.show()

## 3. Municipality-Level Estimates

We estimate probabilities for key Mexican municipalities.

In [ ]:
# Define key municipalities for comparison
municipalities = [
    {'municipality_name': 'Benito Juárez, CDMX', 'total_population': 434_153, 'area_km2': 26.63, 'is_urban': True, 'is_high_edu_state': True, 'marginalization_index': 0.05},
    {'municipality_name': 'Miguel Hidalgo, CDMX', 'total_population': 364_439, 'area_km2': 46.99, 'is_urban': True, 'is_high_edu_state': True, 'marginalization_index': 0.05},
    {'municipality_name': 'Monterrey, NL', 'total_population': 1_135_512, 'area_km2': 325.0, 'is_urban': True, 'is_high_edu_state': True, 'marginalization_index': 0.08},
    {'municipality_name': 'Guadalajara, JAL', 'total_population': 1_385_629, 'area_km2': 151.0, 'is_urban': True, 'is_high_edu_state': True, 'marginalization_index': 0.10},
    {'municipality_name': 'Puebla, PUE', 'total_population': 1_692_181, 'area_km2': 534.0, 'is_urban': True, 'is_high_edu_state': False, 'marginalization_index': 0.20},
    {'municipality_name': 'Mérida, YUC', 'total_population': 892_363, 'area_km2': 858.0, 'is_urban': True, 'is_high_edu_state': False, 'marginalization_index': 0.15},
    {'municipality_name': 'Oaxaca de Juárez, OAX', 'total_population': 264_217, 'area_km2': 88.0, 'is_urban': True, 'is_high_edu_state': False, 'marginalization_index': 0.35},
    {'municipality_name': 'Guerrero Negro, BCS', 'total_population': 12_000, 'area_km2': 2_800, 'is_urban': False, 'is_high_edu_state': False, 'marginalization_index': 0.60},
]

mun_df = pd.DataFrame(municipalities)
results = model.estimate_batch(mun_df)

display(results[['municipality_name', 'population', 'expected_count', 'prob_at_least_one', 'ci_95_low', 'ci_95_high']].round(8))

In [ ]:
# Visualize probabilities
fig = px.bar(
    results.sort_values('prob_at_least_one'),
    x='prob_at_least_one',
    y='municipality_name',
    orientation='h',
    error_x=results.sort_values('prob_at_least_one')['ci_95_high'] - results.sort_values('prob_at_least_one')['prob_at_least_one'],
    labels={'prob_at_least_one': 'P(≥1 prodigy)', 'municipality_name': ''},
    title='Probability of Finding a Sub-18 Graduate by Municipality',
    color='prob_at_least_one',
    color_continuous_scale='YlOrRd',
)
fig.update_layout(template='plotly_white', showlegend=False)
fig.show()

## 4. Radius Query: Danish Embassy, Mexico City

The motivating question: what is P(≥1 prodigy within 1km of the Danish Embassy in Polanco)?

In [ ]:
# Miguel Hidalgo contains Polanco (where Danish Embassy is located)
mh_estimate = model.estimate_municipality(
    municipality_name='Miguel Hidalgo, CDMX',
    population=364_439,
    area_km2=46.99,
    is_urban=True,
    is_high_edu_state=True,
    marginalization_index=0.04,  # Very low marginalization in Polanco
)

radius_estimates = []
for r in [0.5, 1.0, 2.0, 5.0, 10.0]:
    est = model.estimate_radius(mh_estimate, radius_km=r)
    radius_estimates.append({
        'radius_km': r,
        'area_km2': est.area_km2,
        'population_est': est.population,
        'expected_count': est.expected_count,
        'prob_at_least_one': est.prob_at_least_one,
        'ci_low': est.credible_interval_95[0],
        'ci_high': est.credible_interval_95[1],
    })

radius_df = pd.DataFrame(radius_estimates)
display(radius_df.round(8))

In [ ]:
# Plot P vs radius
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=radius_df['radius_km'],
    y=radius_df['prob_at_least_one'],
    mode='lines+markers',
    name='P(≥1 prodigy)',
    line=dict(color='#e63946', width=2),
    marker=dict(size=8),
))
fig.add_trace(go.Scatter(
    x=radius_df['radius_km'],
    y=radius_df['ci_high'],
    fill=None, mode='lines', line_color='rgba(230,57,70,0.2)', showlegend=False
))
fig.add_trace(go.Scatter(
    x=radius_df['radius_km'],
    y=radius_df['ci_low'],
    fill='tonexty', mode='lines', line_color='rgba(230,57,70,0.2)',
    fillcolor='rgba(230,57,70,0.1)', name='95% CI'
))
fig.add_vline(x=1.0, line_dash='dash', line_color='gray', annotation_text='1km radius')
fig.update_layout(
    title='P(Finding Sub-18 Graduate) vs Radius — Danish Embassy Area, CDMX',
    xaxis_title='Radius (km)',
    yaxis_title='Probability',
    yaxis_tickformat='.2e',
    template='plotly_white',
)
fig.show()

## 5. Sensitivity Analysis

In [ ]:
# Vary base rate assumptions
base_rates = [1e-7, 5e-7, 1e-6, 5e-6, 1e-5]
populations = [5_000, 20_000, 50_000, 100_000, 500_000]

sa_df = sensitivity_analysis(base_rates, populations)

fig = px.density_heatmap(
    sa_df,
    x='base_rate_per_million',
    y='population',
    z='prob_at_least_one',
    color_continuous_scale='YlOrRd',
    labels={
        'base_rate_per_million': 'Base rate (per million)',
        'population': 'Population in area',
        'prob_at_least_one': 'P(≥1)',
    },
    title='Sensitivity: P(≥1 Prodigy) by Base Rate Assumption and Population',
)
fig.show()

## 6. Export Results

In [ ]:
results.to_csv('../data/processed/municipality_probabilities.csv', index=False)
radius_df.to_csv('../data/processed/radius_query_danish_embassy.csv', index=False)
print('Results exported to data/processed/')